# Model Load

In [3]:
import os
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, Idefics2ForConditionalGeneration, BitsAndBytesConfig
from peft import PeftConfig, PeftModel

# ==========================================
# 1. 환경 및 경로 설정 (여기를 수정하세요)
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42 # 데이터 분할 시 항상 같은 평가셋을 얻기 위해 시드 고정

# 원본 CSV 파일 및 이미지 폴더 경로
CSV_PATH = "train.csv" 
IMAGE_ROOT = "." 


# ==========================================
# 2. 동일한 평가 데이터셋(eval_df) 고정 추출
# ==========================================
# Seed를 고정했기 때문에, 언제 실행해도 항상 똑같은 100개의 문제가 eval_df에 할당됩니다.
full_train_df = pd.read_csv(CSV_PATH)
train_df = full_train_df.sample(n=200, random_state=SEED).reset_index(drop=True)
remaining_df = full_train_df.drop(train_df.index)
eval_df = remaining_df.sample(n=100, random_state=SEED).reset_index(drop=True)

In [4]:
# ==========================================
# 3. 데이터셋 및 프로세서 설정
# ==========================================
class IdeficsQuizDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['path'])
        image = Image.open(img_path).convert('RGB')
        
        # 4지 선다형 프롬프트
        question = row['question']
        options = f"(a) {row['a']} (b) {row['b']} (c) {row['c']} (d) {row['d']}"
        prompt = f"User: <image>{question}\n{options}\n위 질문의 정답을 a, b, c, d 중 하나로 골라주세요.\nAssistant: 정답은 ("
        
        return {
            "image": image,
            "prompt": prompt,
            "answer": row['answer'].lower().strip()
        }

def quiz_collate_fn(examples):
    images = [ex["image"] for ex in examples]
    prompts = [ex["prompt"] for ex in examples]
    answers = [ex["answer"] for ex in examples]
    
    # 여기서 프로세서가 패딩을 처리하고 텐서로 변환합니다.
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    return inputs, answers

In [ ]:
ADAPTER_PATH = "./qwen3_vl_32b_lora"
print(f"\n[{ADAPTER_PATH}] 모델 로딩 중...")
print(f"평가 데이터 준비 완료: 총 {len(eval_df)} 문항")

# ==========================================
# 4. 모델(Base + Adapter) 및 프로세서 로드
# ==========================================
print(f"\n[{ADAPTER_PATH}] 모델 로딩 중...")
config = PeftConfig.from_pretrained(ADAPTER_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # 4비트 양자화의 표준 타입
    bnb_4bit_compute_dtype=torch.bfloat16, # 연산은 16비트로 진행
    bnb_4bit_use_double_quant=True, # 양자화 파라미터까지 한 번 더 압축
    llm_int8_enable_fp32_cpu_offload=True
)

# Base 모델 로드 (메모리 최적화를 위해 bfloat16 적용)
base_model = AutoModelForVision2Seq.from_pretrained(
    config.base_model_name_or_path, 
    quantization_config=bnb_config,
    device_map="auto" 
)


[./qwen3_vl_32b_lora] 모델 로딩 중...
평가 데이터 준비 완료: 총 100 문항

[./qwen3_vl_32b_lora] 모델 로딩 중...


c:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2\Ai\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]c:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2\Ai\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards:  57%|█████▋    | 8/14 [17:46<13:55, 139.33s/it]c:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2\Ai\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\SSAFY\Downloads\2026-s

RuntimeError: Tensor.item() cannot be called on meta tensors

In [7]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftConfig, PeftModel
import torch

print(f"\n[{ADAPTER_PATH}] 모델 로딩 중...")
config = PeftConfig.from_pretrained(ADAPTER_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True, 
    llm_int8_enable_fp32_cpu_offload=True
)

base_model = AutoModelForImageTextToText.from_pretrained(
    config.base_model_name_or_path, 
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 # 핵심 해결책: 가짜 텐서 버그 방지
)

# 어댑터(LoRA) 병합
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# 프로세서 로드
processor = AutoProcessor.from_pretrained(ADAPTER_PATH)

print("✅ 모델 로드 완벽 성공! 이제 추론을 시작합니다.")


[./qwen3_vl_32b_lora] 모델 로딩 중...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards:   7%|▋         | 1/14 [03:22<43:51, 202.45s/it]


RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 8388608000 bytes.

In [ ]:
# Base 모델에 학습된 Adapter 병합
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.to(DEVICE)
model.eval()

# 프로세서 로드
processor = AutoProcessor.from_pretrained(ADAPTER_PATH)

# DataLoader 생성
eval_dataset = IdeficsQuizDataset(eval_df, IMAGE_ROOT, processor)
eval_loader = DataLoader(eval_dataset, batch_size=4, collate_fn=quiz_collate_fn)

In [ ]:
# ==========================================
# 5. 추론 및 채점
# ==========================================
correct = 0
total = 0
results = []

print("추론 시작...")
with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
    for batch, real_answers in tqdm(eval_loader):
        # 텐서만 필터링하여 GPU로 이동 (안전 장치)
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        # 딱 한 글자(a,b,c,d)만 생성
        generated_ids = model.generate(**batch, max_new_tokens=2)
        generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
        
        for i, full_text in enumerate(generated_texts):
            # "정답은 (" 이후의 첫 글자 추출
            try:
                prediction = full_text.split("정답은 (")[-1][0].lower().strip()
            except:
                prediction = "n/a"
            
            target = real_answers[i]
            is_correct = (prediction == target)
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                "질문": eval_df.iloc[total-1]['question'],
                "예측": prediction,
                "정답": target,
                "결과": "O" if is_correct else "X"
            })

# ==========================================
# 6. 최종 결과 출력 및 저장
# ==========================================
accuracy = (correct / total) * 100
model_name = os.path.basename(os.path.normpath(ADAPTER_PATH))

print(f"\n" + "="*40)
print(f"📊 [{model_name}] 성능 평가 결과")
print(f"전체 문항: {total}")
print(f"맞은 문항: {correct}")
print(f"최종 정확도(Accuracy): {accuracy:.2f}%")
print("="*40)

# (선택) 오답 노트를 포함한 전체 결과를 CSV로 저장
result_df = pd.DataFrame(results)
save_filename = f"eval_result_{model_name}.csv"
result_df.to_csv(save_filename, index=False, encoding='utf-8-sig')
print(f"상세 결과가 '{save_filename}'에 저장되었습니다.")


[./qwen3_vl_32b_lora] 모델 로딩 중...
평가 데이터 준비 완료: 총 100 문항

[./qwen3_vl_32b_lora] 모델 로딩 중...


c:\Users\SSAFY\Downloads\2026-ssafy-ai-15-2\Ai\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
W0403 10:03:35.857000 20492 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 